In [0]:
%sql
--table creation script:
-- Employee table
CREATE TABLE employees (
    emp_id INT,
    emp_name VARCHAR(50),
    dept_id INT,
    manager_id INT,
    salary INT,
    hire_date DATE,
    job_title VARCHAR(50)
);

In [0]:
%sql
INSERT INTO employees VALUES
(1, 'John', 10, NULL, 90000, '2018-01-10', 'Manager'),
(2, 'Sara', 10, 1, 70000, '2019-03-15', 'Developer'),
(3, 'Amit', 20, 5, 60000, '2020-05-20', 'Developer'),
(4, 'Lina', 30, 1, 75000, '2017-09-11', 'Analyst'),
(5, 'Raj', 20, NULL, 95000, '2016-12-05', 'Manager'),
(6, 'Dev', 30, 1, 72000, '2021-04-18', 'Analyst'),
(7, 'Kim', 10, 1, 50000, '2022-02-02', 'Intern'),
(8, 'Jill', 20, 5, 85000, '2019-07-22', 'Lead');

In [0]:
%sql
INSERT INTO employees VALUES
(9, 'Sopan', 20, 5, 100000, '2019-07-22', 'Lead');

In [0]:
%sql
-- Department table
CREATE TABLE departments_1 (
    dept_id INT,
    dept_name VARCHAR(50)
);

In [0]:
%sql
INSERT INTO departments_1 VALUES
(10, 'IT'),
(20, 'Finance'),
(30, 'HR'),
(40, 'Marketing');

In [0]:
%sql
-- Sales table
CREATE TABLE sales_1 (
    sale_id INT,
    product_id INT,
    sale_date DATE,
    amount DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO sales_1 VALUES
(1, 101, '2024-01-01', 1000),
(2, 101, '2024-01-02', 1200),
(3, 101, '2024-01-03', 1500),
(4, 101, '2024-01-04', 800),
(5, 102, '2024-01-01', 700),
(6, 102, '2024-01-03', 1200),
(7, 102, '2024-01-05', 1300);

In [0]:
%sql
select * from employees

In [0]:
%sql
--1Q find 2nd highest salary and using this you can calculate 3rd and 4th highest salary
-- You cannot use a window function (DENSE_RANK()) directly in the WHERE clause in most SQL engines.
-- WHERE is executed before window functions are calculated.
-- so need to use CTE
with cte as(
select *,
dense_rank() over(order by salary desc) as drn
from employees)
select * from cte
where drn=2


In [0]:
%sql
-- Q2 Find dublicate records
select emp_id, count(*)
from employees
group by emp_id
having count(*) > 1

In [0]:
%sql
-- Q3 delete the dublicate records
with cte as (select *,
row_number() over(partition by emp_id order by emp_id) as rn
from  employees)
delete from cte where rn>1;

In [0]:
%sql
-- Q4 Employees Earning more than there manager
select  e.emp_name as employee, m.emp_name as manager
from employees e
join employees m
on e.manager_id = m.emp_id
where e.salary > m.salary

In [0]:
%sql
-- Q5 find emp who join before than manager

select e.emp_name as employee,e.hire_date, m.emp_name as manager, m.hire_date
from employees as e
join employees as m
on e.emp_id = m.manager_id
where e.hire_date < m.hire_date

In [0]:
select * from sales_1


In [0]:
-- Q 6 # Retrieves the 7_Day rolling avarage of the sales-- 
-- select * from sales_1
-- This is the window frame 👇
-- CURRENT ROW → today’s row
-- 6 PRECEDING → previous 6 rows
-- 👉 So total rows considered = 7 rows (6 previous + current)
-- 📊 What it means
-- For each row:
-- It takes current day + last 6 days
-- Calculates average of amount
-- Gives a 7-day rolling average
-- in this current row is 7th row and last is 1st

select *, avg(amount) over(order by sale_date rows between 6 preceding and current row) as roll_avg_7_day
from sales_1

In [0]:
-- Q7 for each product
select *, avg(amount) over(partition by product_id order by sale_date rows between 6 preceding and current row) as roll_avg_7_day
from sales_1

In [0]:
-- Q8 running total of sales
-- by defauilt it is unbounded preceding
select *, sum(amount) over(order by sale_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW ) as running_total
from sales_1

In [0]:
select * from employees

In [0]:
select * from departments_1

In [0]:
-- Q9 top3 salary per department
with cte as (select d.dept_name, e.salary,
dense_rank() over(partition by d.dept_name order by e.salary desc) as rnk
from employees as e
left join departments_1 as d
on e.dept_id= d.dept_id)
select * from cte
where rnk < 4;


In [0]:
-- Q10 find department with no employee
select * 
from departments_1 as d
left join employees as e 
on d.dept_id=e.dept_id
where e.dept_id is null

In [0]:
-- Q11 Emp with same job in mmultiple department
select emp_name, job_title, count(distinct dept_id) as num_dept from employees
group by emp_name, job_title
having num_dept > 1
